# Codex short-story action extraction

Run the configured prompt from `src/llm/config.py` over the selected MuLab stories with Codex CLI. The notebook processes stories sequentially, checkpoints each validated result, and can safely resume an interrupted run.

Before running the preflight cell, replace `PROMPT_NAME` below with the new prompt key you add to `PROMPTS`. That prompt must accept `nl` and return a JSON action array.

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'src').is_dir() and (candidate / 'data').is_dir():
            return candidate
    raise RuntimeError('Could not find the project root containing src/ and data/.')


def find_codex_executable() -> Path:
    """Find Codex even when a Jupyter kernel does not inherit the login-shell PATH."""
    on_path = shutil.which('codex')
    if on_path:
        return Path(on_path).expanduser().absolute()

    candidates = [
        Path.home() / '.local/bin/codex',
        Path.home() / '.npm-global/bin/codex',
    ]
    candidates.extend(Path.home().glob('.nvm/versions/node/*/bin/codex'))
    executable_candidates = [
        candidate.absolute()
        for candidate in candidates
        if candidate.is_file() and os.access(candidate, os.X_OK)
    ]
    if executable_candidates:
        return max(executable_candidates, key=lambda path: path.stat().st_mtime_ns)

    raise FileNotFoundError(
        'Could not find the Codex CLI. Install it or set CODEX_EXECUTABLE '
        'manually in the configuration cell.'
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
CODEX_EXECUTABLE = find_codex_executable()
CODEX_ENV = os.environ.copy()
CODEX_ENV['PATH'] = str(CODEX_EXECUTABLE.parent) + os.pathsep + CODEX_ENV.get('PATH', '')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.llm.config import PROMPTS, generate_prompt

# Set this after adding the new prompt entry to src/llm/config.py.
PROMPT_NAME = 'nl2p_unfiltered'
STORIES_PATH = PROJECT_ROOT / 'data/short_stories/mulab_short_stories/selected_stories.json'
MODEL = 'gpt-5.6-terra'
REASONING_EFFORT = 'medium'
MAX_ATTEMPTS = 3
CALL_TIMEOUT_SECONDS = 15 * 60
RETRY_BACKOFF_SECONDS = 5
STORY_LIMIT = None  # Set to a positive integer for a smoke run.

OUTPUT_DIR = PROJECT_ROOT / 'results' / 'short_stories' / PROMPT_NAME / MODEL
print(f'Project root: {PROJECT_ROOT}')
print(f'Codex executable: {CODEX_EXECUTABLE}')
print(f'Story manifest: {STORIES_PATH}')
print(f'Output directory: {OUTPUT_DIR}')

Project root: /home/char37/Projects/MPhil/llm-action-extraction
Codex executable: /home/char37/.nvm/versions/node/v26.1.0/bin/codex
Story manifest: /home/char37/Projects/MPhil/llm-action-extraction/data/short_stories/mulab_short_stories/selected_stories.json
Output directory: /home/char37/Projects/MPhil/llm-action-extraction/results/short_stories/nl2p_unfiltered/gpt-5.6-terra


In [2]:
def validate_actions(actions: Any) -> None:
    if not isinstance(actions, list):
        raise TypeError('Codex response must be a JSON array.')
    for index, action in enumerate(actions):
        if not isinstance(action, dict) or set(action) != {'verb', 'arguments'}:
            raise ValueError(f'Action {index} must contain exactly verb and arguments.')
        if not isinstance(action['verb'], str) or not action['verb'].strip():
            raise ValueError(f'Action {index} has an invalid verb.')
        if not isinstance(action['arguments'], list) or not all(isinstance(arg, str) for arg in action['arguments']):
            raise ValueError(f'Action {index} has invalid arguments.')


def write_json_atomic(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f'.{path.name}.{uuid.uuid4().hex}.tmp')
    try:
        temporary_path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
        temporary_path.replace(path)
    finally:
        temporary_path.unlink(missing_ok=True)


def extract_usage(event_stream: str) -> dict[str, Any] | None:
    """Return the last usage object exposed by Codex JSONL events, if any."""
    usage = None
    for line in event_stream.splitlines():
        try:
            event = json.loads(line)
        except json.JSONDecodeError:
            continue
        candidates = [event.get('usage')]
        for key in ('item', 'response', 'turn', 'payload'):
            nested = event.get(key)
            if isinstance(nested, dict):
                candidates.append(nested.get('usage'))
        for candidate in candidates:
            if isinstance(candidate, dict):
                usage = candidate
    return usage


def load_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding='utf-8'))


def existing_result_matches(path: Path, story: dict[str, str], prompt: str) -> dict[str, Any] | None:
    if not path.exists():
        return None
    try:
        result = load_json(path)
        validate_actions(result.get('actions'))
        expected = {
            'method': PROMPT_NAME,
            'model': MODEL,
            'reasoning_effort': REASONING_EFFORT,
        }
        if all(result.get(key) == value for key, value in expected.items()):
            return result
    except (OSError, TypeError, ValueError, json.JSONDecodeError):
        pass
    return None


In [3]:
# Validate configurations
if PROMPT_NAME not in PROMPTS:
    raise KeyError(f'Prompt {PROMPT_NAME!r} is not in PROMPTS.')
if PROMPTS[PROMPT_NAME].get('parameters') != ['nl']:
    raise ValueError(f'{PROMPT_NAME!r} must declare parameters exactly as [\"nl\"].')
if MAX_ATTEMPTS < 1 or CALL_TIMEOUT_SECONDS < 1:
    raise ValueError('MAX_ATTEMPTS and CALL_TIMEOUT_SECONDS must be positive.')

# Get avaliable models
catalog = json.loads(
    subprocess.run(
        [str(CODEX_EXECUTABLE), 'debug', 'models'],
        text=True, capture_output=True, check=True, env=CODEX_ENV,
    ).stdout
    )
model_entry = next(
    (entry for entry in catalog.get('models', []) if entry.get('slug') == MODEL),None
    )
if model_entry is None:
    raise RuntimeError(f'{MODEL!r} is not available in the installed Codex catalog.')

# Get avaliable reasoning efforts
available_efforts = {
    item.get('effort') for item in model_entry.get('supported_reasoning_levels', [])
    }
if REASONING_EFFORT not in available_efforts:
    raise RuntimeError(f'{MODEL} does not support reasoning effort {REASONING_EFFORT!r}: {sorted(available_efforts)}')

manifest = load_json(STORIES_PATH)
required_story_keys = {'dataset_title', 'content'}
stories = manifest['stories'][:STORY_LIMIT] if STORY_LIMIT is not None else manifest['stories']
if not stories:
    raise ValueError('No stories selected for this run.')

prepared_stories = []
for index, story in enumerate(stories):
    prompt = generate_prompt(PROMPT_NAME, {'nl': story['content']})
    prepared_stories.append({'story': story, 'prompt': prompt})

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Validated {len(prepared_stories)} selected stories; output: {OUTPUT_DIR}')

Validated 6 selected stories; output: /home/char37/Projects/MPhil/llm-action-extraction/results/short_stories/nl2p_unfiltered/gpt-5.6-terra


In [4]:
def run_story(prepared: dict[str, Any]) -> tuple[str, dict[str, Any], int]:
    story = prepared['story']
    title = story['dataset_title']
    prompt = prepared['prompt']
    output_path = OUTPUT_DIR / f'{title}.json'

    # resumed = existing_result_matches(output_path, story, prompt)
    # if resumed is not None:
    #     return 'resumed', resumed, 0

    last_error: Exception | None = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        raw_path = OUTPUT_DIR / f'.{title}.{uuid.uuid4().hex}.codex.json'
        started = time.perf_counter()
        try:
            command = [
                str(CODEX_EXECUTABLE), 'exec', '--ephemeral', '--ignore-user-config',
                '--model', MODEL,
                '--config', f'model_reasoning_effort=\"{REASONING_EFFORT}\"',
                '--sandbox', 'read-only',
                '--output-last-message', str(raw_path),
                '--json', '--color', 'never', '--cd', str(PROJECT_ROOT), '-',
            ]
            completed = subprocess.run(
                command, input=prompt, text=True, capture_output=True,
                timeout=CALL_TIMEOUT_SECONDS, check=False, env=CODEX_ENV,
            )
            elapsed = time.perf_counter() - started
            if completed.returncode != 0:
                details = completed.stderr.strip() or completed.stdout.strip()
                raise RuntimeError(f'Codex exited with {completed.returncode}: {details[-2000:]}')
            if not raw_path.exists():
                raise RuntimeError('Codex completed without writing its final message.')
            raw_response = raw_path.read_text(encoding='utf-8')
            actions = json.loads(raw_response)
            validate_actions(actions)
            result = {
                'dataset': manifest.get('dataset'),
                'source_url': manifest.get('source_url'),
                'source_manifest': str(STORIES_PATH.relative_to(PROJECT_ROOT)),
                'dataset_title': story['dataset_title'],
                'model': MODEL,
                'reasoning_effort': REASONING_EFFORT,
                'method': PROMPT_NAME,
                'content_words': len(story['content'].split()),
                'action_count': len(actions),
                'actions': actions,
                'raw_response': raw_response,
                'usage': extract_usage(completed.stdout),
                'response_time_seconds': elapsed,
            }
            write_json_atomic(output_path, result)
            return 'completed', result, attempt - 1
        except (OSError, RuntimeError, subprocess.TimeoutExpired, json.JSONDecodeError, TypeError, ValueError) as exc:
            last_error = exc
            if attempt < MAX_ATTEMPTS:
                delay = RETRY_BACKOFF_SECONDS * attempt
                print(f'  Attempt {attempt}/{MAX_ATTEMPTS} failed: {exc}. Retrying in {delay}s...')
                time.sleep(delay)
        finally:
            raw_path.unlink(missing_ok=True)
    raise RuntimeError(f"{story['dataset_title']!r} failed after {MAX_ATTEMPTS} attempts: {last_error}")


run_status = {'completed': 0, 'resumed': 0, 'retries': 0, 'failed': 0}
results = []
for index, prepared in enumerate(prepared_stories, start=1):
    title = prepared['story']['dataset_title']
    print(f'[{index}/{len(prepared_stories)}] {title}')
    try:
        status, result, retries = run_story(prepared)
    except Exception as exc:
        run_status['failed'] += 1
        print(f'FAILED: {title}: {exc}')
        print('Completed per-story results were preserved; results.json was not written.')
        raise
    run_status[status] += 1
    run_status['retries'] += retries
    results.append(result)
    print(f"  {status}: {result['action_count']} actions")

combined_result = {
    'dataset': manifest.get('dataset'),
    'source_url': manifest.get('source_url'),
    'source_manifest': str(STORIES_PATH.relative_to(PROJECT_ROOT)),
    'model': MODEL,
    'reasoning_effort': REASONING_EFFORT,
    'method': PROMPT_NAME,
    'story_count': len(results),
    'total_actions': sum(result['action_count'] for result in results),
    'results': results,
}
write_json_atomic(OUTPUT_DIR / 'results.json', combined_result)
print(f"Finished: {run_status}; total actions: {combined_result['total_actions']}")

[1/6] Hansel And Gretel
  completed: 138 actions
[2/6] Chicken-Licken
  completed: 21 actions
[3/6] Jack And The Beanstalk
  completed: 115 actions
[4/6] The Three Remarks
  completed: 78 actions
[5/6] The Three Little Pigs
  completed: 60 actions
[6/6] The Four Skilful Brothers
  completed: 59 actions
Finished: {'completed': 6, 'resumed': 0, 'retries': 0, 'failed': 0}; total actions: 471


In [5]:
# Post-run verification. Run this only after the extraction cell succeeds.
verified_results = []
for prepared in prepared_stories:
    story = prepared['story']
    path = OUTPUT_DIR / f"{story['dataset_title']}.json"
    result = existing_result_matches(path, story, prepared['prompt'])
    if result is None:
        raise AssertionError(f'Invalid or stale result: {path}')
    verified_results.append(result)

combined = load_json(OUTPUT_DIR / 'results.json')
if [item['dataset_title'] for item in combined['results']] != [item['dataset_title'] for item in verified_results]:
    raise AssertionError('results.json does not preserve manifest order.')
if combined['story_count'] != len(verified_results):
    raise AssertionError('Story count mismatch.')
if combined['total_actions'] != sum(item['action_count'] for item in verified_results):
    raise AssertionError('Action-count total mismatch.')

print(f'''Verified {len(verified_results)} stories and {combined['total_actions']} actions.''');
print(f'''Results: {OUTPUT_DIR / 'results.json'}''')

Verified 6 stories and 471 actions.
Results: /home/char37/Projects/MPhil/llm-action-extraction/results/short_stories/nl2p_unfiltered/gpt-5.6-terra/results.json
